# 01 Data Loading and Quality Audit

## tl;dr

This notebook reads `data/NSFC正式增量采集_2014-2026_去重筛选最终结果.csv` and audits the new master table (`9222 x 69`) for record counts, field completeness, null values, duplicated key identifiers, exact duplicate rows, date parseability, and year-axis quality.

The year-axis definitions are fixed as follows:

- `award_year`: main approval/funding year axis for later analyses by project approval or funding year.
- `conclusion_year`: project conclusion year axis for later analyses by conclusion year.

Cells that write the audit summary and log are retained as manual delivery steps; this update only changes the notebook source and does not run cells that would overwrite external figures, tables, or logs.


## Context & Methods

### Key Assumptions

- The project root is the directory containing `data/NSFC正式增量采集_2014-2026_去重筛选最终结果.csv`; whether the notebook starts from the project root or from `code/`, it first searches upward for that root.
- The audit target for the new master table is 9,222 records and 69 fields, with the field list defined by the delivered master-table structure.
- Field completeness is checked in two layers: first, whether all 69 expected fields are present; second, whether key fields such as year axes, project identifiers, project titles, principal investigators, host institutions, and source links are 100% non-null.
- `award_year` is the main approval/funding year axis; `conclusion_year` is the project conclusion year axis. The year-axis audit records each axis's role, parseability, observed range, and interval continuity.
- Null values in non-critical fields are recorded as audit information and are not treated as failures by themselves; null values in critical fields, duplicated key identifiers, record-count mismatches, or field-count mismatches are treated as failures.


In [ ]:
# ===== 1. Environment Setup and Audit Parameters =====
# Use only the Python standard library and pandas; do not modify any files in the data directory.
# Output-writing cells are retained for manual runs; during validation, do not run cells that overwrite external outputs.

from pathlib import Path
from datetime import datetime

import pandas as pd
from IPython.display import display, Markdown


MAIN_DATA_FILENAME = "NSFC正式增量采集_2014-2026_去重筛选最终结果.csv"
MAIN_DATA_RELATIVE_PATH = Path("data") / MAIN_DATA_FILENAME


def find_project_root() -> Path:
    """Search upward for the project root containing the new master dataset, avoiding path errors when launched from code/."""
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / MAIN_DATA_RELATIVE_PATH).exists():
            return candidate
    raise FileNotFoundError(f"Could not find {MAIN_DATA_RELATIVE_PATH.as_posix()}")


PROJECT_ROOT = find_project_root()
DATA_PATH = PROJECT_ROOT / MAIN_DATA_RELATIVE_PATH
OUTPUT_TABLE = PROJECT_ROOT / "output" / "tables" / "01_data_quality_summary.csv"
OUTPUT_LOG = PROJECT_ROOT / "output" / "logs" / "01_data_audit.md"

# Delivery contract for the new master table.
EXPECTED_RECORD_COUNT = 9222
EXPECTED_COLUMN_COUNT = 69
EXPECTED_SHAPE = (EXPECTED_RECORD_COUNT, EXPECTED_COLUMN_COUNT)

# Year-axis definitions: award_year is the main approval/funding year axis, and conclusion_year is the project conclusion year axis.
YEAR_AXES = {
    "award_year": {
        "role": "main approval/funding year axis",
        "expected_min": 2010,
        "expected_max": 2023,
    },
    "conclusion_year": {
        "role": "project conclusion year axis",
        "expected_min": 2014,
        "expected_max": 2024,
    },
}
AWARD_YEAR_REFERENCE_COLUMNS = ["fiscal_year", "ratify_year"]

# Expected master-table fields, used to check whether fields were unexpectedly removed or renamed.
EXPECTED_COLUMNS = [
    "country",
    "source_system",
    "award_id",
    "project_title",
    "abstract_text",
    "keywords_raw",
    "agency_program",
    "award_year",
    "fiscal_year",
    "start_date",
    "end_date",
    "amount_original",
    "currency",
    "amount_type",
    "pi_name",
    "co_pi_names",
    "recipient_org",
    "knowledge_prod_place",
    "research_object_place",
    "beneficiary_city",
    "outcomes_text",
    "record_url",
    "query_signature",
    "primary_theme",
    "secondary_themes",
    "include_flag",
    "exclude_reason",
    "manual_review_flag",
    "matched_keywords",
    "matched_method_terms",
    "matched_object_terms",
    "matched_performance_terms",
    "dedup_key",
    "include_reason",
    "review_flag",
    "detail_id",
    "approval_no",
    "project_name",
    "project_type",
    "depend_unit",
    "project_admin",
    "support_num",
    "ratify_year",
    "conclusion_year",
    "apply_code",
    "project_keywords",
    "research_start_date",
    "research_end_date",
    "project_abstract_cn",
    "project_abstract_en",
    "conclusion_abstract",
    "has_report",
    "journal_paper_count",
    "conference_paper_count",
    "book_count",
    "reward_count",
    "patent_count",
    "result_total_count",
    "results_list_json",
    "query_matches",
    "query_count",
    "search_fields",
    "search_payload_keywords",
    "search_project_types",
    "search_project_type_codes",
    "search_apply_codes",
    "search_year_filters",
    "source_url",
    "last_seen_at",
]

# Critical fields in the new master table; null values here affect deduplication, year analysis, project identification, or source traceability.
CORE_COMPLETE_FIELDS = [
    "award_id",
    "approval_no",
    "detail_id",
    "dedup_key",
    "award_year",
    "conclusion_year",
    "project_title",
    "project_name",
    "project_type",
    "pi_name",
    "recipient_org",
    "depend_unit",
    "query_signature",
    "record_url",
    "source_url",
]

UNIQUE_KEY_FIELDS = ["award_id", "approval_no", "detail_id", "dedup_key", "source_url"]
STRING_ID_DTYPES = {column: "string" for column in UNIQUE_KEY_FIELDS if column != "source_url"}

print(f"Project root: {PROJECT_ROOT}")
print(f"Main data path: {DATA_PATH.relative_to(PROJECT_ROOT)}")
print(f"Year axes: award_year = {YEAR_AXES['award_year']['role']}; conclusion_year = {YEAR_AXES['conclusion_year']['role']}")


## Data

Read the new master data and generate a basic profile. This does not clean or rewrite the raw data; it only computes audit metrics in memory.


In [ ]:
# ===== 2. Read Master Data and Basic Profile =====
# Read key identifier fields as strings to avoid numeric-format changes affecting uniqueness checks.

df = pd.read_csv(DATA_PATH, dtype=STRING_ID_DTYPES, low_memory=False)

row_count, column_count = df.shape
file_size_bytes = DATA_PATH.stat().st_size

# Generate a concise field profile for quick in-notebook review of field types, nulls, and unique values.
profile_rows = []
for column_name in df.columns:
    series = df[column_name]
    null_count = int(series.isna().sum())
    empty_string_count = int(series.dropna().astype(str).str.strip().eq("").sum())
    missing_like_count = null_count + empty_string_count
    profile_rows.append(
        {
            "field": column_name,
            "dtype": str(series.dtype),
            "null_count": null_count,
            "blank_string_count": empty_string_count,
            "missing_rate": round(missing_like_count / row_count, 4) if row_count else None,
            "unique_count": int(series.nunique(dropna=True)),
        }
    )

field_profile = pd.DataFrame(profile_rows)

print(f"Records read: {row_count}")
print(f"Fields read: {column_count}")
print(f"Read shape: {row_count} x {column_count}")
display(field_profile)


## Results

The following section consolidates all audit rules into a long-form table. `status` values mean:

- `PASS`: Meets the current audit criteria.
- `FAIL`: Fails a hard rule and needs priority handling.
- `WARN`: Requires manual confirmation or explanation, but does not necessarily block later analysis.
- `INFO`: Records facts for transparent data-structure reporting or non-critical null values.


In [ ]:
# ===== 3. Build Quality Audit Details =====
# The audit summary uses a long-table structure for later scripts or manual filtering of FAIL/WARN rows.

quality_rows = []


def add_check(category, item, status, observed_value, expected_value="", affected_rows=0, affected_rate=None, details=""):
    """Append one check result to the quality summary table."""
    quality_rows.append(
        {
            "check_category": category,
            "check_item": item,
            "status": status,
            "observed_value": observed_value,
            "expected_value": expected_value,
            "affected_rows": int(affected_rows) if pd.notna(affected_rows) else 0,
            "affected_rate": affected_rate,
            "details": details,
        }
    )


def missing_like_mask(series: pd.Series) -> pd.Series:
    """Identify pandas nulls and strings that become empty after trimming whitespace."""
    return series.isna() | series.dropna().astype(str).str.strip().eq("").reindex(series.index, fill_value=False)


def normalized_string_series(series: pd.Series) -> pd.Series:
    """Convert identifier fields to trimmed strings for uniqueness checks."""
    return series.astype("string").str.strip()


# 3.1 Input-file, record-count, and field-count checks.
add_check(
    "input_file",
    "main_dataset_exists",
    "PASS" if DATA_PATH.exists() else "FAIL",
    DATA_PATH.relative_to(PROJECT_ROOT).as_posix(),
    MAIN_DATA_RELATIVE_PATH.as_posix(),
    details=f"file size {file_size_bytes} bytes",
)
add_check(
    "volume",
    "main_dataset_shape_equals_9222x69",
    "PASS" if (row_count, column_count) == EXPECTED_SHAPE else "FAIL",
    f"{row_count} x {column_count}",
    f"{EXPECTED_RECORD_COUNT} x {EXPECTED_COLUMN_COUNT}",
    affected_rows=abs(row_count - EXPECTED_RECORD_COUNT),
    affected_rate=round(abs(row_count - EXPECTED_RECORD_COUNT) / EXPECTED_RECORD_COUNT, 6),
    details="The new master-data shape must match the delivery contract.",
)
add_check(
    "volume",
    "record_count_equals_9222",
    "PASS" if row_count == EXPECTED_RECORD_COUNT else "FAIL",
    row_count,
    EXPECTED_RECORD_COUNT,
    affected_rows=abs(row_count - EXPECTED_RECORD_COUNT),
    affected_rate=round(abs(row_count - EXPECTED_RECORD_COUNT) / EXPECTED_RECORD_COUNT, 6),
    details="The new master dataset must contain 9,222 records.",
)
add_check(
    "schema",
    "column_count_equals_69",
    "PASS" if column_count == EXPECTED_COLUMN_COUNT else "FAIL",
    column_count,
    EXPECTED_COLUMN_COUNT,
    affected_rows=abs(column_count - EXPECTED_COLUMN_COUNT),
    details="The new master dataset must contain 69 fields.",
)

# 3.2 Field-presence checks.
actual_columns = list(df.columns)
missing_columns = [column for column in EXPECTED_COLUMNS if column not in actual_columns]
unexpected_columns = [column for column in actual_columns if column not in EXPECTED_COLUMNS]
add_check(
    "schema",
    "expected_columns_present",
    "PASS" if not missing_columns else "FAIL",
    len(EXPECTED_COLUMNS) - len(missing_columns),
    len(EXPECTED_COLUMNS),
    affected_rows=len(missing_columns),
    details="Missing columns: " + ("; ".join(missing_columns) if missing_columns else "none"),
)
add_check(
    "schema",
    "unexpected_columns",
    "INFO" if unexpected_columns else "PASS",
    len(unexpected_columns),
    0,
    affected_rows=len(unexpected_columns),
    details="Unexpected columns: " + ("; ".join(unexpected_columns) if unexpected_columns else "none"),
)

# 3.3 Critical-field completeness checks. Null values in critical fields affect deduplication, year analysis, project identification, or source traceability.
for column in CORE_COMPLETE_FIELDS:
    if column not in df.columns:
        add_check(
            "core_completeness",
            f"{column}_complete",
            "FAIL",
            "column missing",
            "column present with 0 null values",
            affected_rows=row_count,
            affected_rate=1.0,
            details="Critical field is missing, so completeness cannot be validated.",
        )
        continue
    mask = missing_like_mask(df[column])
    missing_count = int(mask.sum())
    add_check(
        "core_completeness",
        f"{column}_complete",
        "PASS" if missing_count == 0 else "FAIL",
        row_count - missing_count,
        row_count,
        affected_rows=missing_count,
        affected_rate=round(missing_count / row_count, 6) if row_count else None,
        details=f"Null/blank-string count for critical field {column}.",
    )

# 3.4 All-field null audit. Nulls in non-critical fields are recorded as INFO to avoid treating extension fields as failures.
for column in df.columns:
    mask = missing_like_mask(df[column])
    missing_count = int(mask.sum())
    if missing_count == 0:
        status = "PASS"
    elif column in CORE_COMPLETE_FIELDS:
        status = "FAIL"
    else:
        status = "INFO"
    add_check(
        "nulls",
        f"{column}_missing_like",
        status,
        missing_count,
        0 if column in CORE_COMPLETE_FIELDS else "record missingness",
        affected_rows=missing_count,
        affected_rate=round(missing_count / row_count, 6) if row_count else None,
        details="missing_like = pandas null or a string that becomes empty after trimming whitespace.",
    )

# 3.5 Critical-field duplicate checks. Only non-empty identifier values are checked so repeated nulls do not mask real ID duplicates.
for key_column in UNIQUE_KEY_FIELDS:
    if key_column not in df.columns:
        add_check(
            "uniqueness",
            f"{key_column}_duplicated_rows",
            "FAIL",
            "column missing",
            "0 duplicated keys",
            affected_rows=row_count,
            affected_rate=1.0,
            details=f"Critical uniqueness field {key_column} is missing.",
        )
        continue
    normalized = normalized_string_series(df[key_column])
    valid_mask = normalized.notna() & normalized.ne("")
    duplicated_mask = valid_mask & normalized.duplicated(keep=False)
    duplicated_rows = int(duplicated_mask.sum())
    duplicated_key_count = int(normalized[duplicated_mask].nunique(dropna=True))
    add_check(
        "uniqueness",
        f"{key_column}_duplicated_rows",
        "PASS" if duplicated_rows == 0 else "FAIL",
        duplicated_rows,
        0,
        affected_rows=duplicated_rows,
        affected_rate=round(duplicated_rows / row_count, 6) if row_count else None,
        details=f"duplicated key count: {duplicated_key_count}",
    )

exact_duplicate_rows = int(df.duplicated().sum())
add_check(
    "uniqueness",
    "exact_duplicate_rows",
    "PASS" if exact_duplicate_rows == 0 else "FAIL",
    exact_duplicate_rows,
    0,
    affected_rows=exact_duplicate_rows,
    affected_rate=round(exact_duplicate_rows / row_count, 6) if row_count else None,
    details="exact duplicate row count.",
)

# 3.6 Year-axis audit. award_year is the main approval/funding year axis, and conclusion_year is the project conclusion year axis.
def audit_year_axis(column, role, expected_min, expected_max):
    add_check(
        "year_axis",
        f"{column}_axis_role",
        "PASS" if column in df.columns else "FAIL",
        column if column in df.columns else "column missing",
        role,
        affected_rows=0 if column in df.columns else row_count,
        affected_rate=0 if column in df.columns else 1.0,
        details=f"{column} is recorded as the {role}.",
    )
    if column not in df.columns:
        add_check(
            "year_axis",
            f"{column}_range",
            "FAIL",
            "column missing",
            f"{expected_min}-{expected_max}",
            affected_rows=row_count,
            affected_rate=1.0,
            details="Year field is missing.",
        )
        return []

    numeric_year = pd.to_numeric(df[column], errors="coerce")
    invalid_count = int(numeric_year.isna().sum())
    valid_years = numeric_year.dropna().astype(int)
    observed_min = int(valid_years.min()) if len(valid_years) else None
    observed_max = int(valid_years.max()) if len(valid_years) else None
    observed_years = sorted(valid_years.unique().tolist())

    add_check(
        "year_axis",
        f"{column}_parseable",
        "PASS" if invalid_count == 0 else "FAIL",
        row_count - invalid_count,
        row_count,
        affected_rows=invalid_count,
        affected_rate=round(invalid_count / row_count, 6) if row_count else None,
        details=f"Number of records where {role} cannot be parsed as a year.",
    )

    range_ok = invalid_count == 0 and observed_min == expected_min and observed_max == expected_max
    add_check(
        "year_axis",
        f"{column}_range",
        "PASS" if range_ok else "WARN",
        f"{observed_min}-{observed_max}",
        f"{expected_min}-{expected_max}",
        affected_rows=invalid_count,
        affected_rate=round(invalid_count / row_count, 6) if row_count else None,
        details=f"{role} observed years: {observed_years}",
    )

    missing_years = [year for year in range(observed_min, observed_max + 1) if year not in observed_years] if observed_years else []
    add_check(
        "year_axis",
        f"{column}_year_continuity",
        "PASS" if not missing_years else "WARN",
        "continuous" if not missing_years else "gaps present",
        "continuous within the observed range",
        affected_rows=0,
        affected_rate=0,
        details="Missing years within the observed range: " + ("; ".join(map(str, missing_years)) if missing_years else "none"),
    )
    return observed_years


year_axis_results = {}
for year_column, axis_config in YEAR_AXES.items():
    year_axis_results[year_column] = audit_year_axis(year_column, axis_config["role"], axis_config["expected_min"], axis_config["expected_max"])

# award_year is the main axis; fiscal_year and ratify_year are used only for consistency checks of synonymous year fields.
award_year_numeric = pd.to_numeric(df["award_year"], errors="coerce") if "award_year" in df.columns else None
for reference_column in AWARD_YEAR_REFERENCE_COLUMNS:
    if reference_column not in df.columns or award_year_numeric is None:
        add_check(
            "year_axis",
            f"{reference_column}_matches_award_year",
            "WARN",
            "unable to check",
            "matches award_year",
            affected_rows=row_count,
            affected_rate=1.0,
            details=f"{reference_column} or award_year column is missing.",
        )
        continue
    reference_year = pd.to_numeric(df[reference_column], errors="coerce")
    mismatch_mask = award_year_numeric.ne(reference_year) | award_year_numeric.isna() | reference_year.isna()
    mismatch_count = int(mismatch_mask.sum())
    add_check(
        "year_axis",
        f"{reference_column}_matches_award_year",
        "PASS" if mismatch_count == 0 else "WARN",
        row_count - mismatch_count,
        row_count,
        affected_rows=mismatch_count,
        affected_rate=round(mismatch_count / row_count, 6) if row_count else None,
        details=f"Number of records where {reference_column} differs from the main approval/funding year axis award_year.",
    )

# 3.7 Supplemental parseability checks for date fields.
for date_column in ["start_date", "end_date", "research_start_date", "research_end_date", "last_seen_at"]:
    if date_column not in df.columns:
        add_check("date", f"{date_column}_parseable", "WARN", "column missing", "parseable date", affected_rows=row_count, affected_rate=1.0)
        continue
    parsed_date = pd.to_datetime(df[date_column], errors="coerce")
    invalid_date_count = int(parsed_date.isna().sum())
    add_check(
        "date",
        f"{date_column}_parseable",
        "PASS" if invalid_date_count == 0 else "WARN",
        row_count - invalid_date_count,
        row_count,
        affected_rows=invalid_date_count,
        affected_rate=round(invalid_date_count / row_count, 6) if row_count else None,
        details=f"min date: {parsed_date.min().date() if invalid_date_count < row_count else 'NA'}; max date: {parsed_date.max().date() if invalid_date_count < row_count else 'NA'}",
    )

quality_summary = pd.DataFrame(quality_rows)

# Sort statuses so failures and warnings are easiest to review first.
status_order = {"FAIL": 0, "WARN": 1, "INFO": 2, "PASS": 3}
quality_summary = quality_summary.sort_values(
    by=["status", "check_category", "check_item"],
    key=lambda s: s.map(status_order) if s.name == "status" else s,
).reset_index(drop=True)

display(quality_summary)


## Takeaways

The final output-writing cell saves the in-memory quality summary and Markdown audit log as project deliverables. To validate notebook logic only, run through the quality-summary display and do not run the output-writing cell.


In [ ]:
# ===== 4. Save Quality Summary CSV and Markdown Audit Log =====
# This cell overwrites the external summary table and log; run it manually only when deliverables need refreshing.

OUTPUT_TABLE.parent.mkdir(parents=True, exist_ok=True)
OUTPUT_LOG.parent.mkdir(parents=True, exist_ok=True)

# Write CSV with utf-8-sig so spreadsheet software can open any retained Chinese fields and notes correctly.
quality_summary.to_csv(OUTPUT_TABLE, index=False, encoding="utf-8-sig")

status_counts = quality_summary["status"].value_counts().to_dict()
fail_count = int(status_counts.get("FAIL", 0))
warn_count = int(status_counts.get("WARN", 0))
info_count = int(status_counts.get("INFO", 0))
pass_count = int(status_counts.get("PASS", 0))

if fail_count > 0:
    overall_status = "FAIL: hard quality issue(s) detected"
elif warn_count > 0:
    overall_status = "WARN: hard checks passed, but review warnings remain"
else:
    overall_status = "PASS: no hard issues or review warnings detected"

# Generate a null-value overview that separates critical from non-critical fields.
null_overview = quality_summary[quality_summary["check_category"].eq("nulls")].copy()
nonnull_missing = null_overview[null_overview["affected_rows"].gt(0)].copy()
core_null_failures = quality_summary[
    quality_summary["check_category"].eq("core_completeness") & quality_summary["status"].eq("FAIL")
]


def year_distribution_table(column):
    """Generate a year-distribution table for displaying both year axes in the log."""
    if column not in df.columns:
        return pd.DataFrame(columns=[column, "record_count"])
    year_values = pd.to_numeric(df[column], errors="coerce").dropna().astype(int)
    return year_values.value_counts().sort_index().rename_axis(column).reset_index(name="record_count")


year_distribution_tables = {column: year_distribution_table(column) for column in YEAR_AXES}


def md_escape(value):
    """Escape pipes and line breaks in Markdown tables so the log remains readable."""
    text = str(value)
    return text.replace("|", "\\|").replace("\n", " ")


def df_to_markdown_table(table: pd.DataFrame, columns=None, max_rows=None) -> str:
    """Generate a small Markdown table manually without relying on tabulate."""
    if columns is not None:
        table = table.loc[:, columns]
    if max_rows is not None:
        table = table.head(max_rows)
    if table.empty:
        return "None."
    header = "| " + " | ".join(map(md_escape, table.columns)) + " |"
    divider = "| " + " | ".join(["---"] * len(table.columns)) + " |"
    body = []
    for _, row in table.iterrows():
        body.append("| " + " | ".join(md_escape(row[column]) for column in table.columns) + " |")
    return "\n".join([header, divider, *body])


key_checks = quality_summary[
    quality_summary["check_item"].isin(
        [
            "main_dataset_shape_equals_9222x69",
            "record_count_equals_9222",
            "column_count_equals_69",
            "expected_columns_present",
            "award_id_duplicated_rows",
            "approval_no_duplicated_rows",
            "detail_id_duplicated_rows",
            "dedup_key_duplicated_rows",
            "source_url_duplicated_rows",
            "exact_duplicate_rows",
            "award_year_axis_role",
            "award_year_parseable",
            "award_year_range",
            "award_year_year_continuity",
            "conclusion_year_axis_role",
            "conclusion_year_parseable",
            "conclusion_year_range",
            "conclusion_year_year_continuity",
            "fiscal_year_matches_award_year",
            "ratify_year_matches_award_year",
        ]
    )
].copy()

noncore_missing_table = nonnull_missing[
    ~nonnull_missing["check_item"].str.replace("_missing_like", "", regex=False).isin(CORE_COMPLETE_FIELDS)
][["check_item", "affected_rows", "affected_rate", "status"]]

audit_time = datetime.now().astimezone().strftime("%Y-%m-%d %H:%M:%S %z")

award_years = year_axis_results.get("award_year", [])
conclusion_years = year_axis_results.get("conclusion_year", [])

markdown_lines = [
    "# 01 Data Loading and Quality Audit Log",
    "",
    f"- Generated at: {audit_time}",
    f"- Project root: `{PROJECT_ROOT}`",
    f"- Main data file: `{DATA_PATH.relative_to(PROJECT_ROOT).as_posix()}`",
    f"- Output summary table: `{OUTPUT_TABLE.relative_to(PROJECT_ROOT).as_posix()}`",
    f"- Overall status: **{overall_status}**",
    "",
    "## Key Findings",
    "",
    f"- Master data shape: {row_count} x {column_count}; expected {EXPECTED_RECORD_COUNT} x {EXPECTED_COLUMN_COUNT}; {'PASS' if (row_count, column_count) == EXPECTED_SHAPE else 'FAIL'}.",
    f"- Field inventory: {len(EXPECTED_COLUMNS)} expected fields; missing columns: {'; '.join(missing_columns) if missing_columns else 'none'}; unexpected columns: {'; '.join(unexpected_columns) if unexpected_columns else 'none'}.",
    f"- Critical-field completeness: {'PASS; all ' + str(len(CORE_COMPLETE_FIELDS)) + ' critical fields have no null/blank strings' if core_null_failures.empty else 'critical-field nulls exist; see summary table'}.",
    f"- Year axes: `award_year` is the main approval/funding year axis; `conclusion_year` is the project conclusion year axis.",
    f"- Year ranges: award_year={min(award_years) if award_years else 'NA'}-{max(award_years) if award_years else 'NA'}; conclusion_year={min(conclusion_years) if conclusion_years else 'NA'}-{max(conclusion_years) if conclusion_years else 'NA'}.",
    f"- `award_id` duplicates: {int(quality_summary.loc[quality_summary['check_item'].eq('award_id_duplicated_rows'), 'affected_rows'].iloc[0]) if not quality_summary.loc[quality_summary['check_item'].eq('award_id_duplicated_rows')].empty else 'NA'} rows.",
    f"- `dedup_key` duplicates: {int(quality_summary.loc[quality_summary['check_item'].eq('dedup_key_duplicated_rows'), 'affected_rows'].iloc[0]) if not quality_summary.loc[quality_summary['check_item'].eq('dedup_key_duplicated_rows')].empty else 'NA'} rows.",
    "",
    "## Status Counts",
    "",
    f"- PASS: {pass_count}",
    f"- WARN: {warn_count}",
    f"- INFO: {info_count}",
    f"- FAIL: {fail_count}",
    "",
    "## Key Check Details",
    "",
    df_to_markdown_table(key_checks[["check_category", "check_item", "status", "observed_value", "expected_value", "affected_rows", "details"]]),
    "",
    "## Year-Axis Distributions",
    "",
    "### award_year: main approval/funding year axis",
    "",
    df_to_markdown_table(year_distribution_tables["award_year"]),
    "",
    "### conclusion_year: project conclusion year axis",
    "",
    df_to_markdown_table(year_distribution_tables["conclusion_year"]),
    "",
    "## Non-Critical Field Null Overview",
    "",
    "The following null values are recorded for transparency and are not treated as hard failures; any critical-field nulls are called out in the key findings above.",
    "",
    df_to_markdown_table(noncore_missing_table, max_rows=80),
    "",
    "## Audit Judgment",
    "",
    "- No hard failures were found for record count, field count, expected-field presence, critical-field completeness, or critical uniqueness fields." if fail_count == 0 else "- Hard failures exist and FAIL items should be addressed first.",
    "- The year audit uses award_year as the main approval/funding year axis and conclusion_year as the project conclusion year axis; other synonymous year fields are used only for consistency checks.",
    "",
]

OUTPUT_LOG.write_text("\n".join(markdown_lines), encoding="utf-8")

print(f"Saved quality summary: {OUTPUT_TABLE.relative_to(PROJECT_ROOT)}")
print(f"Saved audit log: {OUTPUT_LOG.relative_to(PROJECT_ROOT)}")
print(overall_status)


In [ ]:
# ===== 5. Output File Read-Back Validation =====
# Run only after manually running the output-writing cell, to confirm both deliverables are readable.

saved_summary = pd.read_csv(OUTPUT_TABLE, encoding="utf-8-sig")
saved_log_text = OUTPUT_LOG.read_text(encoding="utf-8")

print(f"CSV summary rows: {len(saved_summary)}")
print(f"Markdown log characters: {len(saved_log_text)}")
print("Status counts:")
print(saved_summary["status"].value_counts().to_string())

display(saved_summary.head(15))
display(Markdown("\n".join(saved_log_text.splitlines()[:30])))
